PySpark and SEC EDGAR (Government Financial Data) %md


## Step 1: Set Initial Settings

In [0]:
# Import PySpark SQL functions for data transformations (filtering, aggregating, etc.)
from pyspark.sql import functions as F
from pyspark.sql.types import *  # Import data types for defining schemas (StringType, IntegerType, etc.)
from pyspark.sql.window import Window  # Import for ranking and time-series operations

# Import Python libraries for API calls and parallel processing
import requests  # Make HTTP requests to SEC EDGAR API
from concurrent.futures import ThreadPoolExecutor  # Run multiple API calls in parallel
import time  # Add delays to respect SEC rate limits
from datetime import datetime  # Handle date/time operations

# ========================================
# WHY EXPLICIT CONFIGURATION MATTERS
# ========================================
# Production code principle: Always explicitly set critical configurations, even if they're defaults
# Reasons: (1) Makes code portable across different Spark versions and cluster setups
#          (2) Self-documents your intentions for anyone reading the code
#          (3) Protects against cluster-level overrides that might disable features
# ========================================

# ========================================
# ADAPTIVE QUERY EXECUTION (AQE)
# ========================================
# What it is: Feature that watches your query run and adjusts the execution plan in real-time
# How it works: After each stage completes, Spark collects actual statistics (row counts, data sizes)
#               and compares them to initial estimates. If there's a mismatch, it adjusts:
#               - Merges many small partitions into fewer larger ones (less overhead)
#               - Switches join strategies when one table is smaller than expected
#               - Splits oversized partitions to prevent one worker from being overwhelmed
# Why specify it: Even though it's default in Spark 3.2+, explicit setting ensures it works on
#                 older Spark versions (2.4, 3.0) and overrides any cluster-level disabling
# Why it matters here: SEC filing data is highly skewed (some companies file 100x more than others)
#                      AQE automatically handles this without manual partition tuning
# ========================================

# AQE-specific settings (lines below configure Adaptive Query Execution features)
spark.conf.set("spark.sql.adaptive.enabled", "true")  # Master switch: turns on AQE's real-time optimization engine
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")  # Partition merging: combines 100 tiny data chunks into 5 normal ones (reduces task overhead)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")  # Skew handling: splits Apple's 10K filings across multiple workers instead of overwhelming one (prevents bottlenecks)

# Why enable these specific AQE features?
# 1. Master switch (adaptive.enabled): Without this, the other two settings do nothing - it's the on/off button
# 2. Partition merging (coalescePartitions): SEC data varies wildly - one company might have 5 filings, another 500
#    Without merging, Spark creates too many tiny tasks for small companies, wasting time on coordination
# 3. Skew handling (skewJoin): When joining company info with filings, megacorps like Berkshire Hathaway have 1000x
#    more filings than a small startup. Without skew handling, one worker gets stuck processing Berkshire while
#    others sit idle. This feature splits the work evenly across all workers.

# General Spark settings (not AQE-specific)
spark.conf.set("spark.sql.shuffle.partitions", "200")  # Default partitions for distributed operations (groupBy, joins)

# Display settings for cleaner notebook output
spark.conf.set("spark.sql.repl.eagerEval.enabled", "true")  # Show DataFrames without calling .show()
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", "10")  # Limit automatic preview to 10 rows

print("Setup complete! Spark configured with Adaptive Query Execution enabled.")
print(f"Spark version: {spark.version}")
print(f"AQE Status: {spark.conf.get('spark.sql.adaptive.enabled')}")

## Step 2: Ingest SEC EDGAR Data

In [ ]:

# ════════════════════════════════════════════════════════════════
# STEP 2: INGEST SEC EDGAR DATA
# ════════════════════════════════════════════════════════════════
# Goal: Download recent filing records for a chosen set of
# companies from the SEC's public API and load them into Spark.
#
# This cell is split into four parts:
#   1. Configuration  — constants and company identifiers
#   2. Schema         — the table blueprint Spark will use
#   3. Functions      — reusable building blocks for fetching data
#   4. Run            — calls the functions and shows the result
# ════════════════════════════════════════════════════════════════


# ── PART 1: CONFIGURATION ────────────────────────────────────────────────────
# All important constant values (URLs, limits, identifiers) are here
# so they're easy to find and change without digging through the rest of the code.

URL_BASE_EDGAR    = "https://data.sec.gov/submissions"       # Root URL for the SEC filing API
URL_ALL_COMPANIES = "https://www.sec.gov/files/company_tickers.json"  # SEC list of every registered company

# The SEC EDGAR API requires every request to identify who is making it.
# Without this header the server returns a 403 "Forbidden" error and blocks you.
HEADERS = {"User-Agent": "financial-portfolio DaveDevPortfolio@gmail.com"}

MAX_WORKERS = 5    # How many companies to fetch at the same time (explained in Part 3)
RATE_LIMIT_DELAY = 0.12  # Seconds to pause before each request — keeps me under SEC's 10 req/s cap

# ── COST CONTROL GATE ────────────────────────────────────────────────────────
# Databricks charges based on cluster compute time.  Querying all ~10,000
# SEC-registered companies at once would multiply that cost dramatically.
# This cap limits any bulk mode to 500 companies at most, striking a balance
# between broad coverage and keeping the job affordable.
MAX_COMPANIES_LIMIT = 500


# ════════════════════════════════════════════════════════════════
# CHOOSE YOUR QUERY MODE
# ════════════════════════════════════════════════════════════════
# Set QUERY_MODE to one of three options:
#
#   "SPECIFIC"   ->  Query only the hand-picked CIKs in CIKS_TO_QUERY below.
#                    Best for targeted analysis of a known list of companies.
#
#   "FIRST_500"  ->  Automatically pull the first 500 companies from SEC EDGAR,
#                    further capped by MAX_COMPANIES_LIMIT (so if the cap is 100,
#                    only 100 companies are fetched, not 500).
#                    The SEC orders its list by CIK number (roughly chronological
#                    registration order), so this gives the earliest-registered
#                    companies — not the largest or most prominent.
#                    Good for broad analysis without manual curation.
#
#   "ALL"        ->  Pull every company registered with the SEC (~10,000+),
#                    then cap the result at MAX_COMPANIES_LIMIT to
#                    control Databricks compute costs.
# ════════════════════════════════════════════════════════════════

QUERY_MODE = "SPECIFIC"  # <-- Change this value to switch modes


# ── OPTION 1: SPECIFIC COMPANIES ─────────────────────────────────────────────
# Add or remove entries to query exactly the companies you care about.
#
# HOW TO FIND A COMPANY'S CIK NUMBER:
#   1. Go to https://efts.sec.gov/LATEST/search-index?q=%22company+name%22&dateRange=custom
#      Or simpler: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany
#   2. Type the company name (e.g. "Tesla") in the search box and hit Enter.
#   3. The CIK appears as a 10-digit number next to the company name in the results.
#   4. You can paste the raw number — leading zeros are added automatically below.

CIKS_TO_QUERY = [
    "0000320193",  # Apple
    "0000789019",  # Microsoft
    "0001018724",  # Amazon
    "0001652044",  # Alphabet (Google)
    "0001326801",  # Meta
    "0001045810",  # NVIDIA
    "0000051143",  # IBM
    "0000078814",  # JPMorgan Chase
]


# ── PART 2: SCHEMA ───────────────────────────────────────────────────────────
# A schema is a blueprint that tells Spark the exact column names and data types
# before it reads any data.  Without one, Spark guesses — and sometimes guesses
# wrong (e.g. treating a date as plain text, or an ID number as an integer).
#
# StructType([...])               -> the full blueprint (all columns together)
# StructField(name, type, nullable)
#   name     -> column label in the resulting table
#   type     -> StringType() means "store as plain text"
#   nullable -> True means a missing/null value is allowed in this column

FILINGS_SCHEMA = StructType([
    StructField("cik",             StringType(), True),  # Company ID — kept as text because math is never done on it
    StructField("form",            StringType(), True),  # Filing type, e.g. "10-K" (annual report) or "8-K" (major event)
    StructField("filingDate",      StringType(), True),  # Date stored as text for now; will be cast to a real date in Step 3
    StructField("accessionNumber", StringType(), True),  # SEC's unique tracking ID for each individual filing document
])


# ── PART 3: FUNCTIONS ────────────────────────────────────────────────────────

def resolve_ciks(mode: str, specific_ciks: list) -> list:
    """
    Returns the final list of CIK strings to query, based on the chosen mode.

    - "SPECIFIC"  -> normalizes and returns the hand-picked list.
    - "FIRST_500" -> fetches all SEC-registered companies and returns the first 500,
                     capped at MAX_COMPANIES_LIMIT (whichever is smaller).
    - "ALL"       -> fetches all SEC-registered companies and returns up to MAX_COMPANIES_LIMIT.

    The SEC's company_tickers.json endpoint returns a dict keyed by sequential
    integers ("0", "1", "2" ...), each containing a company's CIK, ticker, and name.
    CIKs in that file are plain integers so they must be zero-padded to 10 digits
    to match the format the filings API expects.
    """
    # .zfill(10) pads the left side of the CIK with zeros until it is exactly 10 characters long
    # (e.g. "320193" becomes "0000320193") — the SEC API requires this exact format for every mode,
    # so normalizing here means the user never has to worry about adding leading zeros manually
    if mode == "SPECIFIC":
        return [str(cik).zfill(10) for cik in specific_ciks]

    # Fetch the full company list from SEC (used for both FIRST_500 and ALL)
    # timeout is 15 seconds
    response = requests.get(URL_ALL_COMPANIES, headers=HEADERS, timeout=15)
    response.raise_for_status()
    companies = response.json()  # Returns a dict: {"0": {cik_str, ticker, title}, "1": {...}, ...}

    # Convert each entry's CIK integer to a zero-padded 10-digit string
    all_ciks = [str(v["cik_str"]).zfill(10) for v in companies.values()]

    if mode == "FIRST_500":
        # min() ensures MAX_COMPANIES_LIMIT is always respected, even if it's set below 500
        limited = all_ciks[:min(500, MAX_COMPANIES_LIMIT)]
    else:  # "ALL" — take everything but cap at MAX_COMPANIES_LIMIT to control Databricks costs
        limited = all_ciks[:MAX_COMPANIES_LIMIT]

    # Logs a summary so I can confirm which mode ran and how many companies were selected vs. the total available — e.g. "[info] Mode 'ALL': 10,842 companies found, using 500 (cap = 500)"
    print(f"[info] Mode '{mode}': {len(all_ciks):,} companies found, using {len(limited):,} (cap = {MAX_COMPANIES_LIMIT})")
    return limited


def fetch_filings(cik: str) -> list:
    """
    Downloads filing records for a single company from the SEC API.

    The SEC EDGAR API returns data in "column format" — one list per field — instead of
    the more common "row format" where each record is a complete object.
    This function converts the SEC's column format into a list of row dictionaries
    that Spark can easily load into a table.

    Returns a list of dicts (one per filing), or an empty list if the request fails.
    """
    time.sleep(RATE_LIMIT_DELAY)  # Pause before every request so I never exceed SEC's rate limit

    try:
        # Build the full URL for this company and make the HTTP request.
        # timeout=10 means "give up after 10 seconds" — prevents hanging if the SEC server is slow.
        url = f"{URL_BASE_EDGAR}/CIK{cik}.json"
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()  # If the server returned an error (4xx/5xx), stop and raise an exception

        # Navigate to the "recent filings" section of the JSON response.
        # .get() with a default of {} means "if this key is missing, return an empty dict" — safe fallback.
        recent = response.json().get("filings", {}).get("recent", {})

        # ── WHY THE SEC DATA LOOKS "STRANGE" ─────────────────────────────────
        # Most APIs return rows:  [{"form": "10-K", "date": "2024-10-30"}, ...]
        # The SEC returns columns: {"form": ["10-K", "8-K", ...], "date": ["2024-10-30", ...]}
        #
        # This is the SEC EDGAR API's design — it compresses data for faster transfer.
        # I must reassemble it into rows manually.
        #
        # Step A — extract each column (safe if a field is missing):
        forms      = recent.get("form",            [])  # List of filing types
        dates      = recent.get("filingDate",      [])  # List of corresponding dates
        accessions = recent.get("accessionNumber", [])  # List of corresponding SEC document IDs
        #
        # Step B — zip() stitches the three lists together side-by-side:
        #   forms[0] + dates[0] + accessions[0]  ->  row 1
        #   forms[1] + dates[1] + accessions[1]  ->  row 2  ...
        # ─────────────────────────────────────────────────────────────────────
        return [
            {"cik": cik, "form": form, "filingDate": date, "accessionNumber": acc}
            for form, date, acc in zip(forms, dates, accessions)
        ]

    except Exception as error:
        print(f"[warn] CIK {cik} failed: {error}")  # Log the problem so one failure doesn't stop everything
        return []


def fetch_all_filings(ciks: list, max_workers: int = MAX_WORKERS) -> list:
    """
    Fetches filings for every company in the list, running multiple requests
    at the same time to save time.

    Uses a ThreadPoolExecutor — a tool that runs several tasks simultaneously on
    the same machine using "threads" (independent workers sharing the same CPU).
    This is different from PySpark, which distributes work across multiple physical
    machines. Here, threads overlap the waiting time while the SEC server responds,
    instead of waiting for each company to finish before starting the next.

    Returns a single flat list of row dicts covering all companies.
    """
    # Creates up to max_workers threads; remaining CIKs queue up and are picked up as threads finish
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        results = list(pool.map(fetch_filings, ciks))  # Each call returns one company's list of filings

    # ── FLATTENING: turning a list-of-lists into one single list ─────────────
    # results looks like: [ [Apple row1, Apple row2], [Microsoft row1], ... ]
    # Spark needs:        [ Apple row1, Apple row2, Microsoft row1, ... ]
    # The double loop below collapses the nested structure into one flat list.
    # ─────────────────────────────────────────────────────────────────────────
    return [row for company in results for row in company]


def build_filings_dataframe(rows: list):
    """Loads a flat list of filing-row dicts into a Spark DataFrame using the predefined schema."""
    return spark.createDataFrame(rows, schema=FILINGS_SCHEMA)


# ── PART 4: RUN ──────────────────────────────────────────────────────────────
# Resolve which CIKs to query based on QUERY_MODE, then fetch and load into Spark.
# All the real logic lives in the functions above — this section stays intentionally short.

active_ciks = resolve_ciks(QUERY_MODE, CIKS_TO_QUERY)      # Determine the final CIK list based on chosen mode
rows        = fetch_all_filings(active_ciks)                # Download filings for all companies in parallel
filings_raw = build_filings_dataframe(rows)                 # Load into Spark with a guaranteed schema

print(f"Fetched {filings_raw.count()} filings across {len(active_ciks)} companies  (mode: {QUERY_MODE})")
filings_raw  # Display a preview (enabled by eagerEval in Step 1)
